In [7]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("delta-minio-inspect")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        ",".join([
            "io.delta:delta-spark_2.12:3.2.0",
            "org.apache.hadoop:hadoop-aws:3.3.4",
            "com.amazonaws:aws-java-sdk-bundle:1.12.262",
        ])
    )
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [16]:
!spark-submit \
  --packages io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262 \
  --conf "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension" \
  --conf "spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog" \
  /opt/notebooks/rltm_bi_pltfrm/jobs/ingestion/ingest_to_bronze.py \
  --run-once

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f6d4c7be-3285-4d5a-9052-193b549c844a;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 466ms :: artifacts dl 17ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 

In [17]:
bronze_path = "s3a://lakehouse/bronze/gold_price_events"

df = spark.read.format("delta").load(bronze_path)
print("row_count =", df.count())
df.orderBy("ingestion_ts", ascending=False).show(20, truncate=False)

row_count = 4
+------+------------+-------------------+--------------------------+-----------+-------+-------+--------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+----------------------------------------------------------------+-----------+------------+----------+-----------+
|symbol|source_name |source_event_ts    |ingestion_ts              |price_usd  |bid_usd|ask_usd|currency|payload_json                                                                                                                                                                                                                                                                                             

In [19]:
df.select(
    "event_id",
    "symbol",
    "source_name",
    "source_event_ts",
    "ingestion_ts",
    "price_usd",
    "currency",
    "api_status"
).orderBy("ingestion_ts", ascending=False).show(20, truncate=False)

+----------------------------------------------------------------+------+------------+-------------------+--------------------------+-----------+--------+----------+
|event_id                                                        |symbol|source_name |source_event_ts    |ingestion_ts              |price_usd  |currency|api_status|
+----------------------------------------------------------------+------+------------+-------------------+--------------------------+-----------+--------+----------+
|98468911ddd2be828f1fe3e7fdaede9cabd2775f1e54ec6ca3bc4e781f320e6c|XAU   |gold_api_com|2026-03-21 16:45:10|2026-03-21 16:45:15.009364|4492.200195|USD     |OK        |
|ca1cf5a1cd49e3e3ce98ec07b9aa8c25488ec056c95d7ac120237baf0c965049|XAU   |gold_api_com|NULL               |2026-03-21 16:41:31.837063|4492.200195|USD     |OK        |
|9405c589e9a6b54c755335e881ce4d5af3fff63a5cdcd9286ebee5bac1c6ebcd|XAU   |gold_api_com|NULL               |2026-03-21 16:07:57.295912|4492.200195|USD     |OK        |
|e2e

In [24]:
from pathlib import Path
import os

SEARCH_ROOT = Path("/opt/notebooks")
matches = list(SEARCH_ROOT.rglob("jobs/ingestion/ingest_to_bronze.py"))

assert matches, "No canonical ingest_to_bronze.py found under /opt/notebooks"

JOB_PATH = matches[0]
REPO_ROOT = JOB_PATH.parents[3]   # repo_root/jobs/ingestion/ingest_to_bronze.py

print("cwd      =", os.getcwd())
print("REPO_ROOT =", REPO_ROOT)
print("JOB_PATH  =", JOB_PATH)

cwd      = /opt/notebooks
REPO_ROOT = /opt
JOB_PATH  = /opt/notebooks/jobs/ingestion/ingest_to_bronze.py


In [25]:
from pathlib import Path
import os

REPO_ROOT = Path("/opt/notebooks/")
JOB_PATH = REPO_ROOT / "jobs" / "ingestion" / "ingest_to_bronze.py"

assert JOB_PATH.exists(), f"Canonical job not found: {JOB_PATH}"

PACKAGES = ",".join([
    "io.delta:delta-spark_2.12:3.2.0",
    "org.apache.hadoop:hadoop-aws:3.3.4",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262",
])

cmd = (
    f'spark-submit '
    f'--packages {PACKAGES} '
    f'--conf "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension" '
    f'--conf "spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog" '
    f'{JOB_PATH} --run-once'
)

print("Executing:", cmd)
os.system(cmd)

Executing: spark-submit --packages io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262 --conf "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension" --conf "spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog" /opt/notebooks/jobs/ingestion/ingest_to_bronze.py --run-once
:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-361bb610-ab34-4c76-9573-d16d036c2825;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 113ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 f

2026-03-21 17:03:21,975 | INFO | ingest_to_bronze | Wrote 1 Bronze row to s3a://lakehouse/bronze/gold_price_events | source=gold_api_com | symbol=XAU | price_usd=4492.200195


0

In [26]:
from pathlib import Path
import os

print("cwd =", os.getcwd())

candidates = list(Path("/opt/notebooks").rglob("ingest_to_bronze.py"))
for p in candidates:
    print(p)

cwd = /opt/notebooks
/opt/notebooks/jobs/ingestion/ingest_to_bronze.py
